<a href="https://colab.research.google.com/github/Abhinav-Kakaraparthi/Abhinav-Kakaraparthi/blob/main/UntitleKPMG_x_Columbia_DSI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q faiss-cpu sentence-transformers transformers accelerate pypdf

import os
import re
import json
import pickle
import numpy as np
import torch
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
import os
import urllib.request

os.makedirs("rag_project/data/raw", exist_ok=True)

urls = [
    "https://arxiv.org/pdf/1706.03762.pdf",
    "https://arxiv.org/pdf/1810.04805.pdf",
    "https://arxiv.org/pdf/1910.01108.pdf"
]

for i, url in enumerate(urls):
    filepath = f"rag_project/data/raw/doc_{i+1}.pdf"
    urllib.request.urlretrieve(url, filepath)
    print("Downloaded:", filepath)

print("Done")

Downloaded: rag_project/data/raw/doc_1.pdf
Downloaded: rag_project/data/raw/doc_2.pdf
Downloaded: rag_project/data/raw/doc_3.pdf
Done


In [ ]:
BASE_DIR = "rag_project"
RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")
PROCESSED_DATA_DIR = os.path.join(BASE_DIR, "data", "processed")
INDEX_DIR = os.path.join(BASE_DIR, "index")

os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(INDEX_DIR, exist_ok=True)

EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"
GEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # swap to 3B if Colab handles it

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
DENSE_TOP_K = 20
RERANK_TOP_K = 5

BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

In [ ]:
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\x00", " ")
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def extract_pdf_pages(pdf_path: str):
    docs = []
    filename = os.path.basename(pdf_path)
    reader = PdfReader(pdf_path)

    for page_num, page in enumerate(reader.pages, start=1):
        txt = clean_text(page.extract_text() or "")
        if txt:
            docs.append({
                "source": filename,
                "page": page_num,
                "text": txt
            })
    return docs

def load_all_pdfs():
    docs = []
    for fname in os.listdir(RAW_DATA_DIR):
        if fname.lower().endswith(".pdf"):
            part = extract_pdf_pages(os.path.join(RAW_DATA_DIR, fname))
            docs.extend(part)
            print(f"Processed {fname}: {len(part)} pages")
    return docs

In [ ]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + chunk_size, n)
        piece = text[start:end].strip()
        if piece:
            chunks.append((piece, start, end))
        if end == n:
            break
        start += chunk_size - overlap

    return chunks

def build_chunks(documents):
    out = []
    for doc in documents:
        pieces = chunk_text(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP)
        for i, (piece, s, e) in enumerate(pieces):
            out.append({
                "chunk_id": f'{doc["source"]}_p{doc["page"]}_c{i}',
                "source": doc["source"],
                "page": doc["page"],
                "text": piece,
                "start_char": s,
                "end_char": e
            })
    return out

In [ ]:
documents = load_all_pdfs()
chunks = build_chunks(documents)

with open(os.path.join(PROCESSED_DATA_DIR, "documents.json"), "w", encoding="utf-8") as f:
    json.dump(documents, f, indent=2)

with open(os.path.join(PROCESSED_DATA_DIR, "chunks.json"), "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2)

print("Documents:", len(documents))
print("Chunks:", len(chunks))
print(chunks[0])

Processed doc_1.pdf: 15 pages
Processed doc_2.pdf: 16 pages
Processed doc_3.pdf: 5 pages
Documents: 36
Chunks: 161
{'chunk_id': 'doc_1.pdf_p1_c0', 'source': 'doc_1.pdf', 'page': 1, 'text': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu Łukasz Kaiser ∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗ ‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder

In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

def embed_passages(texts, batch_size=16):
    vecs = embed_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return vecs.astype("float32")

def embed_queries(queries):
    queries = [BGE_QUERY_PREFIX + q for q in queries]
    vecs = embed_model.encode(
        queries,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return vecs.astype("float32")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
chunk_texts = [c["text"] for c in chunks]
embeddings = embed_passages(chunk_texts)

np.save(os.path.join(INDEX_DIR, "embeddings.npy"), embeddings)
with open(os.path.join(INDEX_DIR, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

faiss.write_index(index, os.path.join(INDEX_DIR, "faiss_index.bin"))
with open(os.path.join(INDEX_DIR, "faiss_metadata.pkl"), "wb") as f:
    pickle.dump(chunks, f)

print("Embeddings shape:", embeddings.shape)
print("FAISS index size:", index.ntotal)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embeddings shape: (161, 768)
FAISS index size: 161


In [ ]:
def dense_retrieve(query, top_k=DENSE_TOP_K):
    qvec = embed_queries([query])
    scores, indices = index.search(qvec, top_k)

    results = []
    for rank, idx in enumerate(indices[0], start=1):
        item = chunks[idx].copy()
        item["dense_score"] = float(scores[0][rank - 1])
        item["rank"] = rank
        results.append(item)
    return results

In [ ]:
reranker = CrossEncoder(RERANKER_MODEL_NAME)

def rerank(query, candidates, top_k=RERANK_TOP_K):
    pairs = [[query, c["text"]] for c in candidates]
    scores = reranker.predict(pairs)

    reranked = []
    for c, s in zip(candidates, scores):
        item = c.copy()
        item["rerank_score"] = float(s)
        reranked.append(item)

    reranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_k]

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

def build_context(results):
    blocks = []
    for i, r in enumerate(results, start=1):
        blocks.append(
            f"[Evidence {i}] File: {r['source']}, Page: {r['page']}\n{r['text']}"
        )
    return "\n\n".join(blocks)

def answer_query(query, top_k_dense=DENSE_TOP_K, top_k_rerank=RERANK_TOP_K, max_new_tokens=220):
    dense = dense_retrieve(query, top_k=top_k_dense)
    best = rerank(query, dense, top_k=top_k_rerank)
    context = build_context(best)

    prompt = f"""You are a precise research assistant.

Answer the question using only the retrieved evidence.
Do not use outside knowledge.
If the evidence does not support the answer, say exactly:
Not enough information in the retrieved context.

Question:
{query}

Retrieved evidence:
{context}

Write a clear, concise answer in plain English.
Then add:
Sources: file names and page numbers
"""

    messages = [
        {"role": "system", "content": "You answer questions strictly from retrieved evidence."},
        {"role": "user", "content": prompt},
    ]

    text = gen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = gen_tokenizer(text, return_tensors="pt").to(gen_model.device)

    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0
        )

    answer = gen_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return answer, best, prompt

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
answer, best, prompt = answer_query("how many encoder blocks are used in attention is all you need paper")

print("ANSWER:\n")
print(answer)

print("\nTOP RERANKED RESULTS:\n")
for r in best:
    print(r["rerank_score"], r["source"], r["page"])
    print(r["text"][:500])
    print("-" * 100)

ANSWER:

The attention mechanism in the "Attention Is All You Need" paper employs multiple encoder blocks. Specifically, the paper mentions:

- **Encoder**: A stack of N = 6 identical layers.
- **Decoder**: Also consists of a stack of N = 6 identical layers.
- **Multi-head attention**: Used in the decoder's sub-layers, employing 8 parallel attention layers (h = 8).

Therefore, the number of encoder blocks used in the attention mechanism is 6.

Sources:
File: doc_1.pdf, Pages: 3, 5, 9

TOP RERANKED RESULTS:

0.28248539566993713 doc_1.pdf 3
sidual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension dmodel = 512. Decoder: The decoder is also composed of a stack of N = 6 identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack. Similar to the encoder, we employ residual connections around each of the sub-layers

In [ ]:
import gradio as gr

def chat_rag(message, history):
    answer, best, prompt = answer_query(message, top_k_dense=20, top_k_rerank=5, max_new_tokens=220)

    sources = []
    for i, r in enumerate(best, start=1):
        sources.append(f"[{i}] {r['source']} page {r['page']}")

    final = answer + "\n\nSources: " + ", ".join(sources)
    return final

demo = gr.ChatInterface(
    fn=chat_rag,
    title="Research Paper RAG Chat",
    description="Ask grounded questions over your arXiv papers."
)

demo.launch(share=True, debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://eba0bf40078eeb4c0c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://eba0bf40078eeb4c0c.gradio.live
